In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [3]:
BASE_DIR=Path("..")/'data'

train_transaction = pd.read_csv(BASE_DIR/'train_transaction.csv')
test_transaction = pd.read_csv(BASE_DIR/'test_transaction.csv')
train_identity = pd.read_csv(BASE_DIR/'train_identity.csv')
test_identity = pd.read_csv(BASE_DIR/'test_identity.csv')

In [4]:
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
test = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')

In [5]:
# Separate features and target
y_train = train['isFraud']
X_train_raw = train.drop(columns=['isFraud', 'TransactionID'])
X_test_raw = test.drop(columns='TransactionID')

In [6]:
# Drop columns with high missingness
missing_threshold = 0.20
cols_to_keep = X_train_raw.columns[X_train_raw.isnull().mean() <= missing_threshold]
X_train_raw = X_train_raw[cols_to_keep]
X_test_raw = X_test_raw[cols_to_keep]

In [7]:
numericals_train = X_train_raw.select_dtypes(include=np.number) # Select all columns with a numerical data type
categoricals_train = X_train_raw.select_dtypes(exclude=np.number) # Exclude all columns with a numerical data type

# Replicate selection on test
numericals_test = X_test_raw.select_dtypes(include=np.number)
categoricals_test = X_test_raw.select_dtypes(exclude=np.number)

# Fit imputers on train
imputer_numeric = SimpleImputer(strategy='median')
imputer_categorical = SimpleImputer(strategy='most_frequent')

X_train_numeric = pd.DataFrame(imputer_numeric.fit_transform(numericals_train), columns=numericals_train.columns)
X_train_categorical = pd.DataFrame(imputer_categorical.fit_transform(categoricals_train), columns=categoricals_train.columns)

# Transform test with train's imputer
X_test_numeric = pd.DataFrame(imputer_numeric.transform(numericals_test), columns=numericals_test.columns)
X_test_categorical = pd.DataFrame(imputer_categorical.transform(categoricals_test), columns=categoricals_test.columns)

X_train_imputed = pd.concat([X_train_numeric, X_train_categorical], axis=1)
X_test_imputed = pd.concat([X_test_numeric, X_test_categorical], axis=1)


In [8]:
X_train_encoded = pd.get_dummies(X_train_imputed, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_imputed, drop_first=True)

# Align test to train columns
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

In [9]:
# Scaling fit on train
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_encoded), columns=X_train_encoded.columns)

In [10]:
# Split before SMOTE to avoid a data leakage
# Forgot to do that BEFORE and got suspicious numbers
X_train_split, X_test, y_train_split, y_test = train_test_split(X_train_scaled, y_train, test_size=0.2, random_state=42, stratify=y_train)

In [11]:
# Apply SMOTE on train_split
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_split, y_train_split)

In [12]:
X_test_scaled = pd.DataFrame(scaler.transform(X_test_encoded), columns=X_test_encoded.columns)

In [13]:
X_test_scaled.head()

,TransactionDT,TransactionAmt,card1,card2,card3,card5,addr1,addr2,C1,C2,...,P_emaildomain_web.de,P_emaildomain_windstream.net,P_emaildomain_yahoo.co.jp,P_emaildomain_yahoo.co.uk,P_emaildomain_yahoo.com,P_emaildomain_yahoo.com.mx,P_emaildomain_yahoo.de,P_emaildomain_yahoo.es,P_emaildomain_yahoo.fr,P_emaildomain_ymail.com
0,2.389081,-0.430993,0.104111,-1.606256,-0.281425,0.644557,-1.267894,0.069833,-0.060586,-0.059933,...,-0.020164,-0.022732,-0.007361,-0.009109,-0.454041,-0.051183,-0.011195,-0.015065,-0.015563,-0.063827
1,2.389090,-0.359702,-1.148040,-1.606256,-0.281425,0.644557,0.076566,0.069833,-0.083047,-0.085795,...,-0.020164,-0.022732,-0.007361,-0.009109,-0.454041,-0.051183,-0.011195,-0.015065,-0.015563,-0.063827
2,2.389100,0.150412,-1.106417,1.350412,-0.281425,0.644557,1.879602,0.069833,-0.090533,-0.085795,...,-0.020164,-0.022732,-0.007361,-0.009109,-0.454041,-0.051183,-0.011195,-0.015065,-0.015563,-0.063827
3,2.389100,0.626866,0.222450,-0.016169,-0.281425,-0.813255,-0.903118,0.069833,-0.068073,-0.085795,...,-0.020164,-0.022732,-0.007361,-0.009109,-0.454041,-0.051183,-0.011195,-0.015065,-0.015563,-0.063827
4,2.389101,-0.280467,1.656599,0.571333,-0.281425,-2.003802,-0.288210,0.069833,-0.060586,-0.059933,...,-0.020164,-0.022732,-0.007361,-0.009109,-0.454041,-0.051183,-0.011195,-0.015065,-0.015563,-0.063827


In [14]:
# Sanity check
print("Before SMOTE:")
print(y_train.value_counts())

print("After SMOTE:")
print(y_train_resampled.value_counts())

print("NA count:")
print(f"NAs in train: {X_train_resampled.isna().sum().sum()}")
print(f"NAs in test: {X_test_scaled.isna().sum().sum()}")

Before SMOTE:
isFraud
0    569877
1     20663
Name: count, dtype: int64
After SMOTE:
isFraud
0    455902
1    455902
Name: count, dtype: int64
NA count:
NAs in train: 0
NAs in test: 0


#### Decision Tree

In [ ]:
clf = DecisionTreeClassifier(criterion='entropy', random_state=67, class_weight='balanced')

clf.fit(X_train_resampled, y_train_resampled)
clf_pred = clf.predict(X_test)

print(classification_report(y_test, clf_pred))

Decision Tree:
              precision    recall  f1-score   support

           0       0.98      0.97      0.98    113975
           1       0.41      0.51      0.45      4133

    accuracy                           0.96    118108
   macro avg       0.69      0.74      0.71    118108
weighted avg       0.96      0.96      0.96    118108



#### Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

rf.fit(X_train_resampled, y_train_resampled)
rf_pred = rf.predict(X_test)

print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99    113975
           1       0.81      0.50      0.62      4133

    accuracy                           0.98    118108
   macro avg       0.90      0.75      0.80    118108
weighted avg       0.98      0.98      0.98    118108



- High precision (81%) means there are few false alarms
- Low recall (50%) means the model is missing half the frauds